# Lab: Pandas minimum on real HN data
Pull 50 HN top stories, group by author, emit JSON-clean records.

In [ ]:
!pip install -q pandas

In [ ]:
import json, urllib.request, numpy as np, pandas as pd
HN = 'https://hacker-news.firebaseio.com/v0'
def _get(u):
    with urllib.request.urlopen(u, timeout=10) as r:
        return json.loads(r.read())
ids = _get(f'{HN}/topstories.json')[:50]
rows = [_get(f'{HN}/item/{i}.json') or {} for i in ids]
df = pd.DataFrame(rows)[['id','by','score','descendants','title','time']]
df = df.astype({'id':'Int64','score':'Int64','descendants':'Int64'})
df['time'] = pd.to_datetime(df['time'], unit='s', errors='coerce')
print(df.dtypes)

by_user = (df.groupby('by', dropna=True)
    .agg(stories=('id','count'), total_score=('score','sum'), mean_score=('score','mean'))
    .sort_values('total_score', ascending=False))
print(by_user.head())

recs = (df.head(5)
    .assign(time=lambda d: d['time'].astype(str))
    .replace({np.nan: None})
    .to_dict('records'))
print(json.dumps(recs, indent=2, default=str))

No LLM call. cost = $0.00 / run.